# Step 3 – MT: সিলেটি লেখা → প্রমিত বাংলা + ইংরেজি (NLLB-200)
**Runtime → Change runtime type → T4 GPU**, তারপর **Run all**। প্রথমবার `QUICK = True` (প্রায় ১০ মিনিট), সব ঠিক থাকলে `QUICK = False`।
Checkpoint আর ফল সেভ হয় Drive এর `sylheti_mt` ফোল্ডারে; disconnect হলে Run all দিলে শেষ checkpoint থেকে চলে।

## 1. Drive, GPU, library

In [ ]:
from google.colab import drive
import os, subprocess, sys
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
QUICK = False        # True = small test run. False = full training.
TRAIN = False        # False = skip training, load the best saved model
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.48.3", "accelerate==1.3.0",
                "sacrebleu==2.4.3", "sentencepiece"], check=True)
import torch
assert torch.cuda.is_available(), "GPU নেই: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. ডেটা (ASR এর মতো একই বাক্য-ভিত্তিক train / val / test)

In [ ]:
import pandas as pd, re, unicodedata
ROOT = "/content/drive/MyDrive/Sylheti_Thesis/Ultimate Thisis Data"
MT_DIR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_mt"; os.makedirs(MT_DIR, exist_ok=True)
RUN = MT_DIR + ("/nllb-quick" if QUICK else "/nllb-sylheti")
def clean(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
c = pd.read_csv(ROOT + "/corpus.csv", encoding="utf-8-sig")
c["src"] = c.sylheti_asr.map(clean)          # same style as the ASR output (no punctuation)
LANGS = {"bn": ("bangla", "ben_Beng"), "en": ("english", "eng_Latn")}
def pairs(df):
    rows = []
    for k, (col, code_) in LANGS.items():
        rows += [{"src": s, "tgt": t, "lang": code_, "excel_row": r} for s, t, r in zip(df.src, df[col], df.excel_row)]
    return pd.DataFrame(rows)
sp = {k: c[c.split == k].reset_index(drop=True) for k in ["train", "val", "test"]}
train_p, val_p = pairs(sp["train"]), pairs(sp["val"])
if QUICK:
    train_p, val_p = train_p.sample(2000, random_state=13), val_p.sample(200, random_state=13)
    sp["test"] = sp["test"].head(100)
print({k: len(v) for k, v in sp.items()}, "| train pairs:", len(train_p), "| val pairs:", len(val_p))

## 3. মডেল + baseline (train করার আগে NLLB কেমন অনুবাদ করে)

In [ ]:
import sacrebleu, numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
BASE = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(BASE, src_lang="ben_Beng")   # Sylheti is written in Bengali script
def translate(m, texts, lang, bs=32, beams=4):
    out = []
    m.eval()
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
        with torch.no_grad():
            ids = m.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(lang), num_beams=beams, max_new_tokens=256)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    return out
def score(hyp, ref):
    return round(sacrebleu.corpus_bleu(hyp, [list(ref)]).score, 2), round(sacrebleu.corpus_chrf(hyp, [list(ref)]).score, 2)
BASE_RES = MT_DIR + ("/baseline_quick.csv" if QUICK else "/baseline_results.csv")
if not os.path.exists(BASE_RES):
    base = AutoModelForSeq2SeqLM.from_pretrained(BASE).to("cuda")
    t = sp["test"]; rows = []
    for k, (col, code_) in LANGS.items():
        b, ch = score(translate(base, t.src.tolist(), code_), t[col])
        rows.append({"system": "NLLB zero-shot", "direction": "syl->" + k, "BLEU": b, "chrF": ch})
    b, ch = score(t.src.tolist(), t.bangla)                 # just copy the Sylheti text as "Bangla"
    rows.append({"system": "copy source", "direction": "syl->bn", "BLEU": b, "chrF": ch})
    pd.DataFrame(rows).to_csv(BASE_RES, index=False); del base; torch.cuda.empty_cache()
print(pd.read_csv(BASE_RES).to_string(index=False))

## 4. Training (দুই দিকের অনুবাদ একই মডেলে; সেরা checkpoint রাখা হয়)

In [ ]:
import json, gc
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, TrainerCallback)
from transformers.trainer_utils import get_last_checkpoint
FINAL = RUN + "/final"
class PairSet(torch.utils.data.Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        tok.tgt_lang = r.lang
        enc = tok(r.src, text_target=str(r.tgt), truncation=True, max_length=256)
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": enc["labels"]}
class StopIfNoBetter(TrainerCallback):
    def __init__(self, wait): self.wait = wait
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        v = (metrics or {}).get("eval_loss")
        if v is None or state.best_metric is None or v < state.best_metric: return
        best_step = int(str(state.best_model_checkpoint).rstrip("/").split("-")[-1])
        if state.global_step - best_step >= self.wait:
            print("no better validation loss for", self.wait, "steps -> stopping"); control.should_training_stop = True
if TRAIN:
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE)
    args = Seq2SeqTrainingArguments(
        output_dir=RUN, per_device_train_batch_size=16, gradient_accumulation_steps=2, per_device_eval_batch_size=32,
        learning_rate=1e-4, optim="adafactor", warmup_steps=30 if QUICK else 200, weight_decay=0.0,
        num_train_epochs=8, max_steps=60 if QUICK else -1, gradient_checkpointing=True,
        eval_strategy="steps", save_strategy="steps", eval_steps=30 if QUICK else 250, save_steps=30 if QUICK else 250,
        save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=10 if QUICK else 50, report_to="none")
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=PairSet(train_p), eval_dataset=PairSet(val_p),
                             data_collator=DataCollatorForSeq2Seq(tok, model=model),
                             callbacks=[StopIfNoBetter(wait=750)])
    last = get_last_checkpoint(RUN) if os.path.isdir(RUN) else None
    print("resuming from", last or "the start")
    trainer.train(resume_from_checkpoint=last)
    best = trainer.state.best_model_checkpoint; del trainer, model; gc.collect(); torch.cuda.empty_cache()
else:
    last = get_last_checkpoint(RUN)
    best = json.load(open(last + "/trainer_state.json")).get("best_model_checkpoint") if last else None
print("best checkpoint:", best)
model = AutoModelForSeq2SeqLM.from_pretrained(best).to("cuda").eval()
model.save_pretrained(FINAL); tok.save_pretrained(FINAL)
print("final model saved ->", FINAL)

## 5. Test: BLEU / chrF (সঠিক সিলেটি লেখা থেকে, আর ASR এর লেখা থেকে = পুরো cascade)

In [ ]:
# free GPU memory left over from training, reload the final model in fp16
import gc, sys, torch
sys.last_type = sys.last_value = sys.last_traceback = None

# Aggressively delete potential memory-hogging variables from globals
for _v in ["trainer", "model", "base"]:
    if _v in list(globals().keys()):
        del globals()[_v]

gc.collect()
torch.cuda.empty_cache()

# Load the final model in fp16
model = AutoModelForSeq2SeqLM.from_pretrained(FINAL, torch_dtype=torch.float16)
model = model.to("cuda").eval()

def translate(m, texts, lang, bs=16, beams=4):
    out, i = [], 0
    bos = tok.convert_tokens_to_ids(lang)
    while i < len(texts):
        batch = list(texts[i:i + bs])
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=256).to("cuda")
        n_new = min(256, enc["input_ids"].shape[1] * 2 + 10)
        try:
            with torch.no_grad():
                ids = m.generate(**enc, forced_bos_token_id=bos,
                                 num_beams=beams, max_new_tokens=n_new)
        except torch.cuda.OutOfMemoryError:
            del enc
            gc.collect()
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            print("out of memory -> batch size", bs)
            continue
        out += tok.batch_decode(ids, skip_special_tokens=True)
        i += len(batch)
    return out

t = sp["test"]
rows = []
for k, (col, code_) in LANGS.items():
    hyp = translate(model, t.src.tolist(), code_)
    b, ch = score(hyp, t[col])
    rows.append({"input": "gold Sylheti text", "direction": "syl->" + k,
                 "sentences": len(t), "BLEU": b, "chrF": ch})
    t = t.assign(**{"pred_" + k: hyp})
t.to_csv(RUN + "/predictions_test.csv", index=False, encoding="utf-8-sig")

# cascade: translate what the ASR model wrote for each test recording
ASR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_asr/whisper-small-sylheti"
for name in ["test_seen", "test_unseen"]:
    f = ASR + "/predictions_" + name + ".csv"
    if QUICK or not os.path.exists(f):
        continue
    ref = c[["excel_row", "bangla", "english"]]
    a = pd.read_csv(f)
    a = a.merge(ref, on="excel_row", how="inner")
    a["asr"] = a.pred.fillna("").map(clean)
    for k, (col, code_) in LANGS.items():
        hyp = translate(model, a.asr.tolist(), code_)
        b, ch = score(hyp, a[col])
        rows.append({"input": "ASR output (" + name + ")", "direction": "syl->" + k,
                     "sentences": len(a), "BLEU": b, "chrF": ch})

res = pd.DataFrame(rows)
res.to_csv(RUN + "/results.csv", index=False)
print("baseline:")
print(pd.read_csv(BASE_RES).to_string(index=False))
print()
print("fine-tuned:")
print(res.to_string(index=False))

## 6. কয়েকটা উদাহরণ

In [ ]:
d = pd.read_csv(RUN + "/predictions_test.csv").sample(5)
for _, r in d.iterrows():
    print("সিলেটি  :", r.src); print("বাংলা (আসল):", r.bangla); print("বাংলা (মডেল):", r.pred_bn)
    print("English (ref):", r.english); print("English (model):", r.pred_en); print()

## 7. নিজে লিখে দেখুন

In [ ]:
text = "আমি অখন বাড়িত যাইমু"   # change this Sylheti sentence
print("বাংলা  :", translate(model, [clean(text)], "ben_Beng")[0])
print("English:", translate(model, [clean(text)], "eng_Latn")[0])

##3.

In [ ]:
model.save_pretrained(FINAL)
tok.save_pretrained(FINAL)
res.to_csv(RUN + "/results.csv", index=False)
t.to_csv(RUN + "/predictions_test.csv", index=False, encoding="utf-8-sig")
from google.colab import drive
drive.flush_and_unmount()
print("saved to Drive")

##9. MT demo: type Sylheti/Bangla -> English (+ standard Bangla)

In [ ]:

import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError
NEED = {"transformers": "4.48.3", "gradio": "5.9.1", "pydantic": "2.10.6",
        "huggingface_hub": "0.28.1", "sentencepiece": "0.2.0"}
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
wrong = [p for p, v in NEED.items() if _ver(p) != v]
if wrong:
    pkgs = [p + "==" + v for p, v in NEED.items()]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
    os.kill(os.getpid(), 9)

from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
import re, unicodedata, torch, gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_mt/nllb-sylheti/final"
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL, src_lang="ben_Beng")
mt = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(dev).eval()

def clean(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def translate(text, lang):
    enc = tok([clean(text)], return_tensors="pt", truncation=True, max_length=256).to(dev)
    with torch.no_grad():
        ids = mt.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(lang),
                          num_beams=4, max_new_tokens=256)
    return tok.batch_decode(ids, skip_special_tokens=True)[0]

def run(text):
    if not text or not text.strip():
        return "", ""
    return translate(text, "eng_Latn"), translate(text, "ben_Beng")

demo = gr.Interface(
    fn=run,
    inputs=gr.Textbox(label="সিলেটি / বাংলা লিখুন", lines=3),
    outputs=[gr.Textbox(label="English"), gr.Textbox(label="প্রমিত বাংলা")],
    title="Sylheti → English / Bangla (NLLB)",
    examples=[["আমি অখন বাড়িত যাইমু"], ["তুমি কিতা খাইছো"]],
    flagging_mode="never")
demo.launch(share=True, debug=False, show_api=False)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_mt/nllb-sylheti"
for root, _, files in os.walk(D):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1e6:9.1f} MB  {p.replace(D, '')}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil
shutil.copytree("/content/drive/MyDrive/Sylheti_Thesis/sylheti_mt",
                "/content/drive/MyDrive/Sylheti_Thesis/sylheti_mt_copy", dirs_exist_ok=True)
print("copied")